# 01 - Hello, Jev

Welcome. In this workshop you will learn **Jev**, the first *System One* model, built by [TypeSafe AI](https://typesafe.ai).

This notebook covers:

1. What Jev is, and how it is different from an LLM
2. Your first API call
3. Reading the response: typed answers, probabilities, usage
4. Speed and cost: why the economics are the headline feature

The series continues in:

| Notebook | Topic |
| --- | --- |
| `02_primitives.ipynb` | The four question types in depth: Choice, Score, Noul, Box |
| `03_confidence.ipynb` | Probability vs confidence, and routing on uncertainty |
| `04_patterns.ipynb` | The four architectural patterns, with a live cost experiment |
| `05_real_world.ipynb` | Two real builds, the sharp edges, and the properties reference |

## What is Jev?

LLMs are designed to produce **text for humans to read**. When your *code* needs a judgment
(route this ticket, score this resume, is this spam?), using an LLM means coercing a
text generator into emitting JSON, then parsing it and hoping the shape holds.

Jev inverts this. It is a **System One model**: it never generates text. You send it

- a **state**: the content to evaluate (a string or any JSON structure), and
- a map of typed **questions** about that state,

and it returns one **typed answer per question**, with a full probability distribution.
No prompt engineering for output format, no parsing, no retries on malformed JSON.
The answer is constrained to the options you supplied, by construction.

> The name comes from Daniel Kahneman's *Thinking, Fast and Slow*: System 1 thinking is
> fast and intuitive, System 2 is slow and deliberate. Jev is built for the fast kind:
> the judgment a knowledgeable person makes in a second given the right context.

The three core question types (a fourth, Box, is in preview and covered in notebook 02):

| Question type | Answers | Returns |
| --- | --- | --- |
| **Choice** | Which of these options? | `choice`, `probabilities`, `confidence` |
| **Score** | Which level on my rubric? | `score`, `legend`, `probabilities`, `confidence` |
| **Noul** | Is this true? | `noul` (probability of yes, 0 to 1) |

One more difference that matters: Jev is trained with **RLCD** (reinforcement learning for
calibrated decisions), not RLHF. Its probabilities are optimized to be *calibrated*: across many
predictions, things it gives 0.8 to should happen about 80% of the time. That is what makes the
numbers safe to branch on.

## Setup

You need Python 3.10+ and an API key from the [TypeSafe console](https://console.typesafe.ai/keys):

```bash
pip install typesafe-sdk        # or: uv add typesafe-sdk
export TYPESAFE_API_KEY="your-key"
```

This repo already has both. The cell below finds the key.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Find the TypeSafe API key: environment first, then any .env file up the tree.
if "TYPESAFE_API_KEY" not in os.environ:
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / ".env").exists():
            load_dotenv(folder / ".env")
            break
assert os.environ.get("TYPESAFE_API_KEY"), "Set TYPESAFE_API_KEY (get a key at console.typesafe.ai/keys)"

import typesafe_sdk

print("typesafe-sdk", typesafe_sdk.__version__, "ready")

typesafe-sdk 0.7.5 ready


## Which models exist?

`GET /v1/models` lists what your account can use. Two aliases today:

- `jev-latest`: the most recent stable release (the SDK default)
- `jev-preview`: the most recent release, stable or not; currently ahead of latest

Versioned IDs like `jev-1.13.0` are also accepted. Pin a version in production if you have
tuned thresholds against it; aliases move when a new release ships.

In [2]:
from typesafe_sdk import TypeSafeClient

with TypeSafeClient() as client:
    for m in client.models.list().models:
        print(f"{m.name:<14} released {m.release_date[:10]}   {m.description}")

jev-latest     released 2026-09-10   The latest iteration of TypeSafe's System One Model: Jev
jev-preview    released 2026-09-10   A preview version of `jev-latest`: should be better in most ways


## Your first call

The scenario: a support ticket arrives. We ask three questions **in one request**:
which team should handle it, how frustrated is the customer, and is it urgent.

Things to notice in the code:

- The question IDs (`department`, `frustration`, `is_urgent`) are **yours**. They are never
  sent to the model; answers come back under the same IDs. So write the full question in
  `instructions` even when the ID looks self-explanatory.
- A `Choice` takes `criteria` as a map of option name to description.
- A `Score` takes `criteria` as an **ordered list** of level descriptions, low to high.
- A `Noul` needs only `instructions` (a question, or a statement to judge).

In [3]:
from typesafe_sdk import Choice, Noul, Score, TypeSafeClient

ticket = (
    "Our API integration started returning 500 errors on every request about "
    "20 minutes ago, and we can't process any customer orders until this is fixed."
)

with TypeSafeClient() as client:  # reads TYPESAFE_API_KEY, defaults to jev-latest
    response = client.system_one(
        state=ticket,
        questions={
            "department": Choice(
                instructions="Which team should handle this?",
                criteria={
                    "billing": "Payment or subscription issues",
                    "technical": "Bugs or integration problems",
                    "sales": "Pricing or account questions",
                },
            ),
            "frustration": Score(
                instructions="How frustrated does the customer appear?",
                criteria=[
                    "Calm, just stating facts",
                    "Frustrated but civil",
                    "Very angry, strong language",
                ],
            ),
            "is_urgent": Noul(
                instructions="The message conveys urgency or time-sensitivity",
            ),
        },
    )

print("team:       ", response.choices["department"].choice)
print("frustration:", response.scores["frustration"].score)
print("urgent:     ", response.nouls["is_urgent"].noul)

team:        technical
frustration: 0.77
urgent:      0.99


## Reading the response

The response object gives you:

- `response.answers`: every answer by ID (plus typed views `response.choices`,
  `response.scores`, `response.nouls`, `response.boxes`)
- `response.model`: the **versioned** model that actually answered (log this!)
- `response.usage`: `input_tokens` and `output_tokens`

Let's look at the raw shape. Notice that the Choice answer carries a probability for
**every** option, not just the winner, and a `confidence` summarizing how peaked that
distribution is. Notebook 03 is entirely about those two numbers.

In [4]:
print(response.model_dump_json(indent=2))

{
  "model": "jev-1.13.0",
  "usage": {
    "input_tokens": 431,
    "output_tokens": 73
  },
  "answers": {
    "department": {
      "type": "choice",
      "choice": "technical",
      "confidence": 1.0,
      "probabilities": {
        "billing": 0.0,
        "technical": 1.0,
        "sales": 0.0
      }
    },
    "frustration": {
      "type": "score",
      "score": 0.77,
      "confidence": 0.66,
      "legend": {
        "0": "Calm, just stating facts",
        "1": "Frustrated but civil",
        "2": "Very angry, strong language"
      },
      "probabilities": {
        "0": 0.23,
        "1": 0.77,
        "2": 0.0
      }
    },
    "is_urgent": {
      "type": "noul",
      "noul": 0.99
    }
  }
}


## Speed and cost

Two properties define where Jev fits:

- **Latency.** The docs quote around 100 ms for most queries. It is fast enough to sit in
  a real-time request path, in front of every LLM call, or inside a UI.
- **Price.** $0.042 per **million** input tokens ($42 per billion). Output tokens are free.
  For comparison, frontier LLM input is typically $1 to $15 per million tokens, so Jev is
  two to three orders of magnitude cheaper per token.

Cheap changes architecture: you can afford to ask speculative questions, screen every
message, score every passage in a corpus. Let's measure both properties live.

In [5]:
PRICE_PER_MTOK = 0.042  # dollars per million input tokens; output tokens are free


def cost_usd(response):
    """What this call cost, in dollars."""
    return response.usage.input_tokens / 1_000_000 * PRICE_PER_MTOK


def show(response):
    """Pretty-print every answer in a response, plus tokens and cost."""
    print(f"model={response.model}  input_tokens={response.usage.input_tokens}  cost=${cost_usd(response):.6f}")
    for qid, a in response.answers.items():
        if a.type == "noul":
            print(f"  {qid:<28} noul={a.noul:.2f}")
        elif a.type == "choice":
            print(f"  {qid:<28} choice={a.choice!r}  confidence={a.confidence:.2f}")
            print(f"  {'':<28} probabilities={a.probabilities}")
        elif a.type == "score":
            print(f"  {qid:<28} score={a.score:.2f}  confidence={a.confidence:.2f}")
            print(f"  {'':<28} probabilities={a.probabilities}")
        else:
            print(f"  {qid:<28} {a}")

In [6]:
import time

from typesafe_sdk import Noul, TypeSafeClient

timings = []
with TypeSafeClient() as client:
    for _ in range(3):
        t0 = time.perf_counter()
        r = client.system_one(
            state="Package arrived with a cracked screen.",
            questions={"damaged": Noul(instructions="Does the message report a damaged item?")},
        )
        timings.append((time.perf_counter() - t0) * 1000)

print("latency per call:", "  ".join(f"{t:.0f} ms" for t in timings), "(includes network from this machine)")
print(f"answer: {r.nouls['damaged'].noul:.2f}")
print(f"tokens: {r.usage.input_tokens} in / {r.usage.output_tokens} out")
print(f"cost of this call: ${cost_usd(r):.8f}")
print(f"calls per dollar at this size: {1 / cost_usd(r):,.0f}")

latency per call: 141 ms  111 ms  96 ms (includes network from this machine)
answer: 0.98
tokens: 281 in / 21 out
cost of this call: $0.00001180
calls per dollar at this size: 84,731


## What Jev is NOT

Keep these boundaries in mind; they come from TypeSafe's own docs:

- **Not a chat model.** It cannot write replies, explanations, or summaries.
- **Not a code generator.** You cannot point Claude Code or Cursor at `model="jev-latest"`.
  Jev goes *inside* the software your coding agent writes.
- **Not a calculator.** Arithmetic, counting, and date math belong in your code
  (notebook 05 demonstrates why).
- **Not a reasoning engine.** Multi-step System 2 tasks should be decomposed into atomic
  questions, composed in code.

The mental model to carry forward: **code owns the workflow, Jev answers the judgment calls.**

## Exercises

1. Change the `department` criteria descriptions so that `technical` excludes outages and add
   an `infrastructure` option. Does the answer move?
2. Add a speculative question (`"mentions_refund"`) to the first call. Note the token cost of
   the extra question.
3. Send the same ticket in German. Compare the answers (English is Jev's primary training
   language; see notebook 05 for the properties reference).